In [1]:
import gcsfs
import xarray as xr

In [2]:
fs = gcsfs.GCSFileSystem()
mapper = fs.get_mapper('leap-persistent/jbusecke/scale-aware-air-sea/results/CM26_fluxes_smoothed_v0.6.2test.zarr')
out_mapper = fs.get_mapper('leap-scratch/jbusecke/scale-aware-air-sea/testing/simple_ops.zarr')
out_mapper_long = fs.get_mapper('leap-scratch/jbusecke/scale-aware-air-sea/testing/simple_ops_long.zarr')

In [3]:
def open_zarr(mapper, chunks={}):
    return xr.open_dataset(
        mapper, 
        engine='zarr',
        chunks=chunks,
        consolidated=True,
        inline_array=True
    )

ds = xr.open_dataset(
        mapper,
        engine='zarr',
        chunks={},
        consolidated=True,
        inline_array=True
    )

In [4]:
ds_out = ds.mean(['xt_ocean', 'yt_ocean'])
ds_out

<xarray.Dataset>
Dimensions:    (algo: 2, smoothing: 7, time: 300)
Coordinates:
  * algo       (algo) <U5 'ncar' 'ecmwf'
  * smoothing  (smoothing) <U23 'smooth_none' 'smooth_tracer' ... 'smooth_all'
  * time       (time) object 0181-01-01 12:00:00 ... 0181-10-27 12:00:00
Data variables:
    evap       (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    qh         (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    ql         (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    taux       (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    tauy       (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>

In [5]:
from dask_gateway import Gateway
gateway = Gateway()

# close existing clusters
open_clusters = gateway.list_clusters()
print(list(open_clusters))
if len(open_clusters)>0:
    for c in open_clusters:
        cluster = gateway.connect(c.name)
        cluster.shutdown()
print('setting up new cluster')

options = gateway.cluster_options()
options.worker_memory = 52
options.worker_cores = 4

options.environment = dict(
    DASK_DISTRIBUTED__SCHEDULER__WORKER_SATURATION="1.0",
)

# Create a cluster with those options
cluster = gateway.new_cluster(options)
client = cluster.get_client()
cluster.scale(10)
client

[ClusterReport<name=prod.00312228d9544b9fab98aab708afc2a5, status=RUNNING>, ClusterReport<name=prod.1133eb046f504267985f8e696f6b86f4, status=RUNNING>]
setting up new cluster


Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /services/dask-gateway/clusters/prod.12ea6f5070364b12a08bc1bab25f4700/status,


In [17]:
ds_out.to_zarr(out_mapper, mode='w')

This works perfectly, so the reading the data is not the problem?


Now lets try to build in a delay to simulate a really long running task. This might be the reason the other operation trips up.


In [6]:
# lets construct a mean with a built in an artifical delay
import time
import numpy as np

def long_mean_ufunc(data):
    time.sleep(3*60)
    return np.mean(data, axis=(-2,-1))

ds_out_long = xr.apply_ufunc(long_mean_ufunc, ds, input_core_dims=[['xt_ocean', 'yt_ocean']], dask="parallelized", output_dtypes=[ds.ql.dtype])
ds_out_long

<xarray.Dataset>
Dimensions:    (algo: 2, smoothing: 7, time: 300)
Coordinates:
  * algo       (algo) <U5 'ncar' 'ecmwf'
  * smoothing  (smoothing) <U23 'smooth_none' 'smooth_tracer' ... 'smooth_all'
  * time       (time) object 0181-01-01 12:00:00 ... 0181-10-27 12:00:00
Data variables:
    evap       (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    qh         (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    ql         (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    taux       (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>
    tauy       (algo, smoothing, time) float32 dask.array<chunksize=(1, 1, 3), meta=np.ndarray>

In [ ]:
ds_out_long.to_zarr(out_mapper_long, mode='w')